# Data Collection & Cleaning
This notebook loads and cleans the three CSV datasets needed for the European defense investment project.
At the end, each dataset is filtered to European countries only and merged into one file.

## Import Libraries

In [ ]:
import requests
import pandas as pd
import time

## Define European Countries
A single reference list used to filter all three datasets consistently.
If a country name differs between files, fix it here once rather than in three places.

In [ ]:
EUROPEAN_COUNTRIES = [
    "Albania", "Austria", "Belarus", "Belgium", "Bosnia and Herzegovina",
    "Bulgaria", "Croatia", "Cyprus", "Czechia", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany",
    "Greece", "Hungary", "Iceland", "Ireland", "Italy",
    "Kosovo", "Latvia", "Lithuania", "Luxembourg", "Malta",
    "Moldova", "Montenegro", "Netherlands", "North Macedonia", "Norway",
    "Poland", "Portugal", "Romania", "Russia", "Serbia",
    "Slovakia", "Slovenia", "Spain", "Sweden", "Switzerland",
    "Turkey", "Ukraine", "United Kingdom"
]

---
## Dataset 1: SIPRI Top 100 Defense Companies
We need Company, Country, and Total Revenue.
The final output will be one row per country with the top 3 companies
listed as a string, plus the average revenue of those top 3 companies.

In [ ]:
# Load the raw file
SIPRI_df = pd.read_csv("SIPRI_Companies_2024_Clean.csv")

# Keep only the three columns we care about
SIPRI_df = SIPRI_df[["Company", "Country", "Total revenues_2024"]].copy()

# Rename to a cleaner column name
SIPRI_df = SIPRI_df.rename(columns={"Total revenues_2024": "Total_Revenue_2024_M"})

# Revenue values have commas e.g. "71,040" — remove them so we can treat as numbers
SIPRI_df["Total_Revenue_2024_M"] = SIPRI_df["Total_Revenue_2024_M"].str.replace(",", "", regex=False)
SIPRI_df["Total_Revenue_2024_M"] = pd.to_numeric(SIPRI_df["Total_Revenue_2024_M"], errors="coerce")

# Filter to European companies only
SIPRI_europe = SIPRI_df[SIPRI_df["Country"].isin(EUROPEAN_COUNTRIES)].copy()

# Sort by revenue so the highest-revenue companies come first within each country
SIPRI_europe = SIPRI_europe.sort_values("Total_Revenue_2024_M", ascending=False).reset_index(drop=True)

# .groupby().head(3) keeps only the top 3 rows per country group.
# Because we sorted by revenue above, those 3 rows are the top 3 earners.
top3 = SIPRI_europe.groupby("Country").head(3).reset_index(drop=True)

# Aggregate to one row per country — two separate aggregations, then merge them.
# agg() applies a function to each group. ', '.join combines the company names into one string.
companies_col = top3.groupby("Country")["Company"].agg(", ".join).reset_index()
companies_col.columns = ["Country", "Top_3_Companies"]

# .mean() computes the average revenue across the top 3 companies for each country
revenue_col = top3.groupby("Country")["Total_Revenue_2024_M"].mean().reset_index()
revenue_col.columns = ["Country", "Avg_Top3_Revenue_M"]

# Combine the two aggregations into one clean dataframe
SIPRI_clean = pd.merge(companies_col, revenue_col, on="Country")

print(f"Countries with SIPRI data: {len(SIPRI_clean)}")
print(SIPRI_clean.to_string())

---
## Dataset 2: Military Expenditure as % of GDP (World Bank)
This file has one column per year going back to 1960. We only need the average
across recent years (2015-2024), computed as a single column.

In [ ]:
# Load the raw file
MILXPND_df = pd.read_csv("API_MS_MIL_XPND_GD_ZS_DS2_en_csv_v2_16.csv")

# Build a list of the year columns we want to average: 2015 through 2024
year_cols = [str(y) for y in range(2015, 2025)]

# Keep only Country Name and those year columns
MILXPND_df = MILXPND_df[["Country Name"] + year_cols].copy()
MILXPND_df = MILXPND_df.rename(columns={"Country Name": "Country"})

# Convert year columns to numbers (they may have been read as strings)
MILXPND_df[year_cols] = MILXPND_df[year_cols].apply(pd.to_numeric, errors="coerce")

# Compute the 10-year average across all year columns.
# axis=1 means "average across the columns in each row" (one value per country).
# Missing years are skipped automatically so partial data still gives a valid average.
MILXPND_df["Avg_MilExp_Pct_GDP"] = MILXPND_df[year_cols].mean(axis=1)

# Keep only Country and the average, then filter to Europe
MILXPND_clean = MILXPND_df[["Country", "Avg_MilExp_Pct_GDP"]].copy()
MILXPND_clean = MILXPND_clean[MILXPND_clean["Country"].isin(EUROPEAN_COUNTRIES)].copy()

print(f"European countries: {len(MILXPND_clean)}")
MILXPND_clean.sort_values("Avg_MilExp_Pct_GDP", ascending=False).head(10)

---
## Dataset 3: GDP Growth (World Bank)
We only need Country and average GDP growth.

In [ ]:
# Load the raw file
GDP_df = pd.read_csv("GDP_percent.csv")

# Rename all columns so they are clean to work with
GDP_df.columns = ["Country", "GDP_Growth_2010_2020", "GDP_Growth_2020_2024", "GDP_Avg_Growth"]

# Keep only the two columns we care about
GDP_df = GDP_df[["Country", "GDP_Avg_Growth"]].copy()

# Convert to numeric in case any values were read as strings
GDP_df["GDP_Avg_Growth"] = pd.to_numeric(GDP_df["GDP_Avg_Growth"], errors="coerce")

# Filter to European countries
GDP_europe = GDP_df[GDP_df["Country"].isin(EUROPEAN_COUNTRIES)].copy()

print(f"European countries: {len(GDP_europe)}")
GDP_europe.head(10)

---
## Dataset 4: Web Scrape — Military Expenditure in USD Millions (Trading Economics)

This page has an HTML table with each European country's military spending in USD millions.
That's a different unit from the World Bank % of GDP data, so it adds useful context.

`pd.read_html()` is a pandas function that finds all HTML tables on a page and returns
them as a list of dataframes. Since there's only one table on this page, we take index [0].
This is much simpler than using BeautifulSoup for a straightforward table like this.

In [ ]:
url = "https://tradingeconomics.com/country-list/military-expenditure?continent=europe"

# requests.get() fetches the raw HTML from the URL.
# We pass a User-Agent header so the site doesn't block us as a bot.
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

# pd.read_html() parses all tables in the HTML and returns a list.
# [0] grabs the first (and only) table on the page.
tables = pd.read_html(response.text)
scrape_df = tables[0]

print("Raw scraped columns:", scrape_df.columns.tolist())
scrape_df.head()

In [ ]:
# Keep only Country and the most recent value
scrape_df = scrape_df[["Country", "Last"]].copy()
scrape_df = scrape_df.rename(columns={"Last": "MilExp_USD_Millions"})

# Convert spending to numeric
scrape_df["MilExp_USD_Millions"] = pd.to_numeric(scrape_df["MilExp_USD_Millions"], errors="coerce")

# Align any name differences with our standard country list
scrape_df["Country"] = scrape_df["Country"].str.strip()
scrape_df["Country"] = scrape_df["Country"].replace("Macedonia", "North Macedonia")

# Filter to our European countries list
scrape_df = scrape_df[scrape_df["Country"].isin(EUROPEAN_COUNTRIES)].reset_index(drop=True)

print(f"Countries scraped: {len(scrape_df)}")
scrape_df.sort_values("MilExp_USD_Millions", ascending=False).head(10)

---
## Dataset 5: API Call — Country Coordinates (Open-Meteo Geocoding)

We call the Open-Meteo geocoding API once per country to get its latitude and longitude.
These coordinates are what Folium uses to place each country on the map.

The endpoint takes a country name and returns a JSON object. We pull `latitude` and
`longitude` from the first result. `time.sleep(0.1)` adds a small pause between
requests so we don't hammer the API.

In [ ]:
base_url = "https://geocoding-api.open-meteo.com/v1/search"

coords_list = []  # collect one dictionary per country

for country in EUROPEAN_COUNTRIES:
    # Build the query parameters — requests formats these into a URL automatically
    params = {"name": country, "count": 1, "language": "en", "format": "json"}
    response = requests.get(base_url, params=params)

    # .json() converts the response text into a Python dictionary
    data = response.json()

    # "results" is a list inside the response — take the first entry if it exists
    if "results" in data and len(data["results"]) > 0:
        result = data["results"][0]
        coords_list.append({
            "Country": country,
            "Latitude": result["latitude"],
            "Longitude": result["longitude"]
        })
    else:
        # Store NaN if nothing came back so the row still exists in the final dataframe
        coords_list.append({"Country": country, "Latitude": None, "Longitude": None})
        print(f"No result for: {country}")

    time.sleep(0.1)

coords_df = pd.DataFrame(coords_list)

print(f"Coordinates retrieved: {len(coords_df)}")
coords_df.head(10)

---
## Merge All Five Datasets

`pd.merge()` combines dataframes by matching rows on a shared column — here `"Country"`.
We use `how="outer"` for the first merge so no country is dropped, then `how="left"` for
each addition so every country row is preserved even if a source has no data for it.

In [ ]:
# Start with the two World Bank datasets
merged_df = pd.merge(MILXPND_clean, GDP_europe, on="Country", how="outer")

# Add SIPRI top 3 companies and average revenue
merged_df = pd.merge(merged_df, SIPRI_clean, on="Country", how="left")

# Add scraped USD spending figures
merged_df = pd.merge(merged_df, scrape_df, on="Country", how="left")

# Add coordinates from the API
merged_df = pd.merge(merged_df, coords_df, on="Country", how="left")

# Sort by average military expenditure descending
merged_df = merged_df.sort_values("Avg_MilExp_Pct_GDP", ascending=False).reset_index(drop=True)

print(f"Rows: {len(merged_df)}")
print(f"Columns: {merged_df.columns.tolist()}")
merged_df.head(10)

## Fix Missing Data
Fill all NaN values in numeric columns with 0. Countries with no company data will
rank at the bottom for those categories, which is a reasonable signal for investment.

In [ ]:
# Define the numeric columns that may have NaN values
numeric_cols = ["Avg_MilExp_Pct_GDP", "GDP_Avg_Growth", "MilExp_USD_Millions", "Avg_Top3_Revenue_M"]

# fillna(0) replaces every NaN in those columns with 0
merged_df[numeric_cols] = merged_df[numeric_cols].fillna(0)

print("Missing values after fix:")
print(merged_df[numeric_cols].isnull().sum())

## Rank and Score Each Country

For each scoring category we assign every country a rank from 1 (lowest) to 39 (highest).
`.rank()` does this automatically — `ascending=True` means the smallest value gets rank 1
and the largest gets rank 39, which is what we want since higher values are better in all
four categories.

`method='min'` handles ties by giving both countries the lower rank (e.g. two countries
tied for 5th both get 5, not 5 and 6). We then sum all four rank columns into a single
`Total_Score` — the country with the highest total is the best overall investment candidate.

In [ ]:
# Rank each category — higher value = higher rank (better score)
merged_df["Rank_MilExp_Pct"]   = merged_df["Avg_MilExp_Pct_GDP"].rank(ascending=True, method="min").astype(int)
merged_df["Rank_GDP_Growth"]   = merged_df["GDP_Avg_Growth"].rank(ascending=True, method="min").astype(int)
merged_df["Rank_MilExp_USD"]   = merged_df["MilExp_USD_Millions"].rank(ascending=True, method="min").astype(int)
merged_df["Rank_Avg_Revenue"]  = merged_df["Avg_Top3_Revenue_M"].rank(ascending=True, method="min").astype(int)

# Sum all four ranks into a single composite score
merged_df["Total_Score"] = (
    merged_df["Rank_MilExp_Pct"] +
    merged_df["Rank_GDP_Growth"] +
    merged_df["Rank_MilExp_USD"] +
    merged_df["Rank_Avg_Revenue"]
)

# Drop the intermediate rank columns — we only needed them to compute Total_Score
merged_df = merged_df.drop(columns=["Rank_MilExp_Pct", "Rank_GDP_Growth", "Rank_MilExp_USD", "Rank_Avg_Revenue"])

# Sort by Total_Score to see the top investment candidates
merged_df = merged_df.sort_values("Total_Score", ascending=False).reset_index(drop=True)

print("Top 10 investment candidates:")
print(merged_df[["Country", "Avg_MilExp_Pct_GDP", "GDP_Avg_Growth", "MilExp_USD_Millions", "Avg_Top3_Revenue_M", "Total_Score"]].head(10).to_string())

---
## Save to CSV

In [ ]:
merged_df.to_csv("europe_defense_data.csv", index=False)
print("Saved europe_defense_data.csv")
print(merged_df.to_string())